# 23 — Datos para la página pública: la consulta por tipo de proceso y lugar
La página pública (`despliegue/sitio/`) deja de pedir números y pasa a responder la pregunta de quien tiene un caso:
**«en mi tipo de proceso y en mi ciudad, ¿cuánto tarda la fila y va a crecer?»**. La persona elige materia, tipo de
proceso y lugar; la página busca los datos del Anuario 2023 y muestra:

1. **La fila hoy:** cuánto tardarían los juzgados en terminar las causas que quedaron pendientes al cierre de 2023 si
   siguen resolviendo al ritmo de 2023: `pendientes al cierre ÷ resueltas × 365` días. Es la `duracion_estimada_dias` del
   notebook 10; no necesita modelo.
2. **¿La fila va a crecer el año siguiente?** El modelo de 5 variables del notebook 22, con lo que se sabe **al empezar
   2024**: el stock heredado es lo que quedó pendiente al cierre de 2023, los juzgados son los mismos y, como los ingresos
   de 2024 todavía no se conocen, se usan los de 2023.

Este notebook arma ese escenario, calcula la probabilidad con el modelo guardado (`models/modelo_5var.joblib`) y escribe
`despliegue/sitio/datos/procesos.json`. La página recalcula cada probabilidad con los árboles del JSON del modelo y la
compara con la de Python.

**Advertencia que la página repite:** el escenario de 2024 no se pudo validar, porque solo hay un Anuario (2023). Se mide
abajo cuánto cambian los ingresos de un año a otro en la serie nacional para saber qué tan razonable es usar los del
año anterior.

In [1]:
%run -i modulos/comun.ipynb
%run -i modulos/modelado.ipynb
import joblib

datos = cargar_conjunto()
curado = pd.read_parquet(CURATED / "features" / "dataset_analitico_curado.parquet")
curado = curado[(curado["tipo_elemento_analitico"] == "proceso") & (curado["atendidas"] > 0)].reset_index(drop=True)
assert len(curado) == len(datos) == 1240
assert (curado["tipo_proceso"].to_numpy() == datos["tipo_proceso"].to_numpy()).all()
final = joblib.load(RAIZ / "models" / "modelo_5var.joblib")
TOP5 = json.loads((MODELO_DIR / "top5.json").read_text(encoding="utf-8"))["variables"]
print("variables del modelo:", TOP5)

variables del modelo: ['ingresos_por_juzgado', 'presion_stock', 'stock_por_juzgado', 'materia_homologada', 'ingresos_totales']


## El escenario del año siguiente
Se cambian solo las entradas: **stock heredado ← pendientes al cierre de 2023**; ingresos y juzgados, los de 2023. Las
variables de ingeniería se recalculan con las mismas fórmulas (`construir_variables`).

In [2]:
base = curado[["materia_homologada", "tipo_proceso", "territorio", "departamento_derivado", "ambito",
               "pendientes_inicio_efectivo", "ingresos_totales", "resueltas", "pendientes_fin",
               "num_juzgados_pagina"]].copy()
for c in ["pendientes_inicio_efectivo", "ingresos_totales", "resueltas", "pendientes_fin", "num_juzgados_pagina"]:
    base[c] = base[c].astype("float64")
base["materia_homologada"] = base["materia_homologada"].astype("object")

siguiente = base[["materia_homologada", "ingresos_totales", "num_juzgados_pagina"]].copy()
siguiente["pendientes_inicio_efectivo"] = base["pendientes_fin"]
siguiente = pd.concat([siguiente, construir_variables(siguiente.assign(
    personal_items_acefalias=np.nan, personal_items_total=np.nan, recibidas_excusa_recusacion=np.nan))], axis=1)

p_2023 = final.predict_proba(datos[TOP5])[:, 1]
p_sig = final.predict_proba(siguiente[TOP5])[:, 1]
base["p_siguiente"] = p_sig

bandas = pd.cut(p_sig, [0, 0.4, 0.6, 1], right=False, labels=["probable que no crezca (< 40 %)",
                                                              "incierto (40-60 %)", "probable que crezca (>= 60 %)"])
print(pd.Series(bandas).value_counts().sort_index())
print("correlación entre la probabilidad de 2023 y la del año siguiente:", round(np.corrcoef(p_2023, p_sig)[0, 1], 3))
creció = base["pendientes_fin"] > base["pendientes_inicio_efectivo"]
print("probabilidad media del año siguiente: si la fila creció en 2023", round(p_sig[creció].mean(), 3),
      "· si no creció", round(p_sig[~creció].mean(), 3))

probable que no crezca (< 40 %)    409
incierto (40-60 %)                 423
probable que crezca (>= 60 %)      408
Name: count, dtype: int64
correlación entre la probabilidad de 2023 y la del año siguiente: 0.809
probabilidad media del año siguiente: si la fila creció en 2023 0.549 · si no creció 0.407


## La fila de hoy
`pendientes_fin ÷ resueltas × 365`. Si no se resolvió nada en 2023 la cuenta no existe (se muestra «sin dato»); si no
quedó nada pendiente, la fila es cero.

In [3]:
dias = np.where(base["resueltas"] > 0, base["pendientes_fin"] / base["resueltas"].where(base["resueltas"] > 0) * 365, np.nan)
assert np.allclose(dias[~np.isnan(dias)], curado["duracion_estimada_dias"].to_numpy(dtype=float)[~np.isnan(dias)])
print("procesos sin resueltas (sin dato de fila):", int(np.isnan(dias).sum()))
print("procesos sin pendientes al cierre (fila cero):", int((base["pendientes_fin"] == 0).sum()))
print("mediana de la fila:", round(np.nanmedian(dias)), "días")

procesos sin resueltas (sin dato de fila): 118
procesos sin pendientes al cierre (fila cero): 151
mediana de la fila: 201 días


## ¿Qué tan estables son los ingresos de un año a otro?
Serie nacional 2007-2023 del capítulo 9 (notebook 15). Es el único dato de varios años: por tipo de proceso no hay serie.

In [4]:
serie = pd.read_csv(PROCESSED / "causas_serie_historica.csv")
nac = serie[serie["ambito"] == "nacional"].sort_values("gestion").set_index("gestion")["ingresadas"]
cambio = nac.pct_change().mul(100).dropna()
variacion = {"mediana_pct": round(float(cambio.abs().median()), 1), "max_pct": round(float(cambio.abs().max()), 1),
             "desde": int(cambio.index.min()), "hasta": int(cambio.index.max()),
             "saltos": {str(a): round(float(v), 1) for a, v in cambio[cambio.abs() >= 15].items()}}
print(variacion)

{'mediana_pct': 6.3, 'max_pct': 32.9, 'desde': 2008, 'hasta': 2023, 'saltos': {'2013': 26.9, '2016': -19.0, '2020': -27.1, '2021': 32.9}}


## Nombres para la gente y exportación
Las materias y los lugares se escriben como los conoce cualquiera («Familia», «El Alto», «Provincias de La Paz») y los
tipos de proceso pasan de mayúsculas a oración. Cada fila del JSON es una lista corta para que el archivo pese poco.

In [5]:
MATERIA_CORTA = {
    "Público Civil y Comercial": "Civil y comercial",
    "Público de Familia": "Familia",
    "Público Niñez y Adolescencia": "Niñez y adolescencia",
    "Partido de Trabajo y Seguridad Social": "Trabajo y seguridad social",
    "Partido Administrativo Coactivo Fiscal y Tributario": "Coactivo fiscal y tributario",
}
CIUDAD = {"SUCRE": "Sucre", "LA PAZ": "La Paz", "EL ALTO": "El Alto", "COCHABAMBA": "Cochabamba", "ORURO": "Oruro",
          "POTOSI": "Potosí", "TARIJA": "Tarija", "SANTA CRUZ": "Santa Cruz", "TRINIDAD": "Trinidad", "COBIJA": "Cobija"}
ACRONIMOS = {"afps": "AFP", "rejaps": "REJAPS", "art.": "Art."}


def lugar(r):
    if r["ambito"] == "capital":
        return CIUDAD[r["territorio"]] + (" (ciudad)" if r["territorio"] in ("LA PAZ", "SANTA CRUZ", "COCHABAMBA",
                                                                                 "ORURO", "POTOSI", "TARIJA") else "")
    return "Provincias de " + r["departamento_derivado"]


def tipo_legible(t):
    t = " ".join(t.split()).replace(" -", " – ").replace(" - ", " – ")
    palabras = [ACRONIMOS.get(p.lower(), p.lower()) for p in t.split(" ")]
    s = " ".join(palabras)
    return s[0].upper() + s[1:]


base["lugar"] = base.apply(lugar, axis=1)
base["materia"] = base["materia_homologada"].map(MATERIA_CORTA)
base["tipo"] = base["tipo_proceso"].map(tipo_legible)
assert base["materia"].notna().all()
assert not base.duplicated(["materia", "tipo", "lugar"]).any(), "dos filas con la misma consulta"

materias = list(MATERIA_CORTA.values())
lugares_df = (base[["lugar", "departamento_derivado", "ambito"]].drop_duplicates()
              .sort_values(["departamento_derivado", "ambito", "lugar"]))
lugares = lugares_df["lugar"].tolist()
tipos = (base[["materia", "tipo"]].drop_duplicates().assign(m=lambda d: d["materia"].map(materias.index))
         .sort_values(["m", "tipo"]))
tipo_id = {(r.materia, r.tipo): k for k, r in enumerate(tipos.itertuples())}

filas = []
for r in base.itertuples():
    filas.append([materias.index(r.materia), tipo_id[(r.materia, r.tipo)], lugares.index(r.lugar),
                  int(r.pendientes_inicio_efectivo), int(r.ingresos_totales), int(r.resueltas), int(r.pendientes_fin),
                  int(r.num_juzgados_pagina), float(r.p_siguiente)])

salida = {
    "descripcion": "Procesos no penales del Anuario Estadístico Judicial 2023 con movimiento en el año, para la consulta "
                   "de la página pública. Generado por notebooks/23_sitio_publico.ipynb.",
    "anio_datos": 2023, "anio_proyeccion": 2024,
    "materias": [{"nombre": m, "original": o} for o, m in MATERIA_CORTA.items()],
    "lugares": [{"nombre": r.lugar, "departamento": r.departamento_derivado, "ambito": r.ambito}
                for r in lugares_df.itertuples()],
    "tipos": [{"materia": int(r.m), "nombre": r.tipo} for r in tipos.itertuples()],
    "columnas": ["materia", "tipo", "lugar", "stock_inicio", "ingresos", "resueltas", "pendientes_fin", "juzgados",
                 "p_python"],
    "filas": filas,
    "variacion_ingresos": variacion,
}
ruta = RAIZ / "despliegue" / "sitio" / "datos" / "procesos.json"
ruta.parent.mkdir(parents=True, exist_ok=True)
ruta.write_text(json.dumps(salida, ensure_ascii=False, separators=(",", ":")), encoding="utf-8")
print(len(filas), "filas ·", len(lugares), "lugares ·", len(tipos), "tipos ·", round(ruta.stat().st_size / 1024), "KB")
print(lugares)

1240 filas · 19 lugares · 89 tipos · 59 KB
['Trinidad', 'Provincias de Beni', 'Sucre', 'Provincias de Chuquisaca', 'Cochabamba (ciudad)', 'Provincias de Cochabamba', 'El Alto', 'La Paz (ciudad)', 'Provincias de La Paz', 'Oruro (ciudad)', 'Provincias de Oruro', 'Cobija', 'Provincias de Pando', 'Potosí (ciudad)', 'Provincias de Potosí', 'Santa Cruz (ciudad)', 'Provincias de Santa Cruz', 'Tarija (ciudad)', 'Provincias de Tarija']


## Control
Lo que la página recibe tiene que reconstruir lo que el modelo vio: la materia original, los números y la probabilidad.

In [6]:
J = json.loads(ruta.read_text(encoding="utf-8"))
originales = [m["original"] for m in J["materias"]]
for k in np.random.default_rng(SEMILLA).choice(len(base), 50, replace=False):
    f = J["filas"][k]
    e = pd.DataFrame([{"materia_homologada": originales[f[0]], "ingresos_totales": float(f[4]),
                       "num_juzgados_pagina": float(f[7]), "pendientes_inicio_efectivo": float(f[6])}])
    e = pd.concat([e, construir_variables(e.assign(personal_items_acefalias=np.nan, personal_items_total=np.nan,
                                                   recibidas_excusa_recusacion=np.nan))], axis=1)
    assert abs(final.predict_proba(e[TOP5])[0, 1] - f[8]) < 1e-12
    assert J["tipos"][f[1]]["materia"] == f[0]
print("controles OK: 50 filas al azar reconstruyen la probabilidad exacta")

controles OK: 50 filas al azar reconstruyen la probabilidad exacta


## Lectura
- **La consulta cubre 1.240 procesos en 19 lugares** (las 10 capitales, El Alto incluido, y las provincias de los 9
  departamentos) y 89 combinaciones de materia y tipo de proceso («Guarda» y «Otros procesos» existen en dos materias).
  El archivo pesa 59 KB.
- **La fila de hoy:** mediana de 201 días. En 151 procesos no quedó nada pendiente (fila cero) y en 118 no se resolvió
  ninguna causa en 2023, así que la cuenta no existe: la página dice «sin dato» en vez de inventar un plazo.
- **El escenario de 2024 reparte los procesos en tres tercios:** 408 con probabilidad de 60 % o más («probable que
  crezca»), 423 entre 40 % y 60 % («incierto») y 409 por debajo de 40 %. La página muestra la banda, no solo el número,
  porque el modelo acierta alrededor de 6 de cada 10.
- **La fila tiende a seguir como venía:** la probabilidad del año siguiente se parece a la de 2023 (correlación 0,81) y es
  más alta donde la fila ya creció en 2023 (0,55 contra 0,41).
- **Usar los ingresos del año anterior es razonable en años normales:** en la serie nacional los ingresos cambian una
  mediana de 6,3 % al año. Pero hubo saltos: +26,9 % en 2013, −19,0 % en 2016, −27,1 % en 2020 y +32,9 % en 2021. En un
  año así la proyección falla, y por tipo de proceso no hay serie para medirlo.
- **No está validado entre años.** Para saber cuánto acierta de verdad la proyección hace falta un segundo Anuario con el
  mismo detalle (2022 o 2024): entrenar con un año y evaluar con el siguiente.
